# Confidence vs accuracy and consistency

Figs 2, 6 and 11–16 of the paper. Final models of the main runs, test split.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from notebook_helper import *

trained = load_trained(split="test")        # trained[dataset][model][probe] -> runs
baseline = load_baseline()                  # verbalised confidence of the base models
acc, cons = load_performance(split="test")  # per-question accuracy and consistency


def run_corrs(target, d, m, probe):
    """r(target, trained confidence) of every run of (dataset, model, probe)."""
    return np.array([nan_pearsonr(target[d][m], run["test"]["predictions"])
                     for run in trained[d][m][probe]])

## Figs 2 and 6 — Accuracy vs consistency under a train → test topic shift
Left: Qwen3.5-9B, one point per question. Right: Δr = r(conf, accuracy) − r(conf, consistency), mean ± SEM over models.

In [ ]:
FS = dict(base=19, title=21, label=21, tick=18, legend=18, annot=18, header=23, sub=19)
MODEL = "Qwen3.5-9B"
SCATTER_DATASETS = ["medmcqa4_categoriesbalanced", "math4_categoriesbalanced"]
SHIFT = {"math4": "similar", "math10": "similar",   # test topics vs training topics
         "mmlupro4": "different", "mmlupro10": "different", "medmcqa4": "different"}
SHIFT_STYLE = {"similar": ("Similar test topics", COLORS["red"]),
               "different": ("Different test topics", COLORS["navy"])}
PRETTY = {"medmcqa4": "MedMCQA", "math4": "MATH", "math10": "MATH-10",
          "mmlupro4": "MMLU-Pro", "mmlupro10": "MMLU-Pro-10"}


def header(hax, title, subtitle, color):
    """Column header: bold title, grey subtitle, thin rule below."""
    hax.set_axis_off()
    hax.set_xlim(0, 1)
    hax.set_ylim(0, 1)
    hax.axhline(0.0, color="0.6", lw=1.0)
    hax.text(0.5, 0.68, title, ha="center", va="center", fontsize=FS["header"],
             weight="bold", color=color)
    hax.text(0.5, 0.24, subtitle, ha="center", va="center", fontsize=FS["sub"], color="0.3")


def scatter_fit(ax, x, y, color, xlabel, ylabel, legend=False):
    """Questions coloured by density, quadratic fit, r and R² in the title."""
    xf, yf = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    density_scatter(ax, xf, yf, s=5, alpha=0.6, label="data")
    r2 = quadratic_fit(ax, x, y, color=color, lw=2.5, label="quadratic fit")
    ax.set_title(f"Pearson r = {nan_pearsonr(x, y):.2f}   R² = {r2:.2f}",
                 fontsize=FS["annot"], pad=6)
    if legend:
        leg = ax.legend(loc="lower right", markerscale=3, framealpha=0.9)
        for h in leg.legend_handles:
            h.set_alpha(1)
            if isinstance(h, PathCollection):      # solid swatch for the points
                h.set_array(None)
                h.set_facecolor(COLORS["points"])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    style_ax(ax)

In [ ]:
def plot_shift(probe, probes_right, name):
    """Left: scatters of `probe`. Right: Δr of `probes_right` (one probe: bars coloured by sign)."""
    set_style(FS)
    fig = plt.figure(figsize=(17, 11.5), layout="constrained")
    fig.get_layout_engine().set(wspace=0.02, hspace=0.01)
    gs = fig.add_gridspec(3, 5, width_ratios=[1, 0.04, 1, 0.04, 1.05], height_ratios=[0.30, 1, 1])

    # Left: confidence vs accuracy (top) and vs consistency (bottom), one column per dataset
    for j, d in enumerate(SCATTER_DATASETS):
        conf = trained[d][MODEL][probe][0]["test"]["predictions"]
        ylabel = "confidence" if j == 0 else " "
        ax_acc, ax_cons = fig.add_subplot(gs[1, 2 * j]), fig.add_subplot(gs[2, 2 * j])
        scatter_fit(ax_acc, acc[d][MODEL], conf, COLORS[probe], "accuracy", ylabel)
        scatter_fit(ax_cons, cons[d][MODEL], conf, COLORS[probe], "consistency", ylabel,
                    legend=(j == 1))
        for ax in (ax_acc, ax_cons):            # same ticks everywhere, 1.0 always shown
            ax.set_ylim(ax.get_ylim()[0], max(ax.get_ylim()[1], 1.03))
            ax.set_xlim(ax.get_xlim()[0], max(ax.get_xlim()[1], 1.03))
            ax.yaxis.set_major_locator(MultipleLocator(0.2))
            ax.xaxis.set_major_locator(MultipleLocator(0.25))
        title, color = SHIFT_STYLE[SHIFT[SHORT_LABELS[d]]]
        header(fig.add_subplot(gs[0, 2 * j]), title,
               f"{PRETTY[SHORT_LABELS[d]]}   ·   {MODEL} probe", color)

    # Right: Δr per dataset, similar-topic datasets on top
    order = sorted(DATASETS, key=lambda d: SHIFT[SHORT_LABELS[d]] != "similar")
    short = np.array([SHORT_LABELS[d] for d in order])
    groups = np.array([SHIFT[s] for s in short])
    delta = {p: np.array([[np.nanmean(run_corrs(acc, d, m, p) - run_corrs(cons, d, m, p))
                           for m in MODELS] for d in order])        # (datasets, models)
             for p in probes_right}
    means = {p: np.nanmean(delta[p], axis=1) for p in probes_right}
    errs = {p: sem(delta[p], axis=1) for p in probes_right}

    ax = fig.add_subplot(gs[1:, 4])
    header(fig.add_subplot(gs[0, 4]), f"Average over {len(MODELS)} models",
           "mean ± SEM   ·   Δr > 0: tracks accuracy more", "0.15")
    yk = np.arange(len(order))
    offsets, slot = bar_offsets(len(probes_right), group_width=0.7)
    ax.axhline((groups == "similar").sum() - 0.5, color="0.6", lw=1.0, ls="--")
    ax.axvline(0, color="black", lw=0.8, zorder=2)
    for p, off in zip(probes_right, offsets):
        color = (COLORS[p] if len(probes_right) > 1 else
                 [COLORS["red"] if v > 0 else COLORS["steel"] for v in means[p]])
        ax.barh(yk + off, means[p], height=0.88 * slot, color=color, edgecolor="black",
                zorder=3, label=PROBE_LABELS[p])
        ax.errorbar(means[p], yk + off, xerr=errs[p], fmt="none", ecolor="black",
                    elinewidth=1.2, capsize=3, capthick=1.2, zorder=4)
    lo = min(0, min(np.nanmin(means[p] - errs[p]) for p in probes_right))
    hi = max(0, max(np.nanmax(means[p] + errs[p]) for p in probes_right))
    ax.set_xlim(lo - 0.08 * (hi - lo), hi + 0.08 * (hi - lo))
    ax.set_ylim(len(order) - 0.5, -0.5)
    ax.set_yticks(yk)
    ax.set_yticklabels(short)
    ax.yaxis.tick_right()
    ax.tick_params(axis="y", length=0)
    for lab in ax.get_yticklabels():            # datasets of the left panels in bold
        if lab.get_text() in [SHORT_LABELS[d] for d in SCATTER_DATASETS]:
            lab.set_fontweight("bold")
    style_ax(ax, grid_axis="x", hide=("top", "left"))
    ax.set_xlabel("Δr = r(conf, accuracy) − r(conf, consistency)")
    if len(probes_right) > 1:
        ax.legend(loc="lower right", frameon=True, framealpha=0.9, handlelength=1.4,
                  handleheight=0.9)
    save_fig(fig, name)
    plt.show()

    # Δr per group of datasets: average over the group per model, then mean ± SEM over models
    for p in probes_right:
        for g in ("similar", "different"):
            v = np.nanmean(delta[p][groups == g], axis=0)
            print(f"{p:<4} {g:<10} {np.nanmean(v):+.4f} ± {sem(v):.4f}   "
                  f"({', '.join(short[groups == g])})")

In [ ]:
plot_shift("end", ["end"], "fig02_shift_end")

In [ ]:
plot_shift("mid", ["end", "mid"], "fig06_shift_mid")

## Figs 11 and 12 — Confidence before and after training
Pearson r between confidence and accuracy: verbalised confidence (baseline) and trained probes, mean ± SEM over runs.

In [ ]:
def plot_bars(probes, name):
    """One panel per model, bars per dataset; returns the bar heights {series: (models, datasets)}."""
    fs = dict(base=15, title=16, label=17, tick=14, legend=16)
    set_style(fs)
    series = [("baseline", "baseline")] + [(p, PROBE_LABELS[p]) for p in probes]
    ks = np.arange(len(DATASETS), dtype=float)
    offsets, slot = bar_offsets(len(series), group_width=0.62)
    bars = {key: [] for key, _ in series}

    fig, axs = plt.subplots(2, 5, figsize=(16, 7.4), sharex=True, sharey=True, layout="constrained")
    for i, (m, ax) in enumerate(zip(MODELS, axs.flat)):
        for (key, label), off in zip(series, offsets):
            if key == "baseline":
                means = np.array([nan_pearsonr(acc[d][m], baseline[d][m]) for d in DATASETS])
                errs = None
            else:
                corrs = [run_corrs(acc, d, m, key) for d in DATASETS]
                means = np.array([np.nanmean(c) for c in corrs])
                errs = np.array([sem(c) for c in corrs])
            ax.bar(ks + off, means, yerr=errs, width=0.88 * slot, color=COLORS[key],
                   edgecolor="black", zorder=3, error_kw=ERR_KW, label=label)
            bars[key].append(means)
        ax.set_title(m.replace("-Instruct", "-Inst"), pad=8)
        ax.axhline(0, color="black", lw=0.8, zorder=2)
        style_ax(ax)
        ax.tick_params(axis="x", length=0)
        if i % 5:
            ax.tick_params(axis="y", length=0)
        ax.set_xlim(ks[0] - 0.5, ks[-1] + 0.5)

    axs[0, 0].set_xticks(ks)
    axs[0, 0].set_yticks(np.arange(0, 1.01, 0.2))
    axs[0, 0].set_ylim(top=1.04)
    for ax in axs[-1]:
        ax.set_xticklabels([SHORT_LABELS[d] for d in DATASETS], rotation=30, ha="right",
                           rotation_mode="anchor")
    fig.legend(*axs[0, 0].get_legend_handles_labels(), loc="outside lower center",
               ncols=len(series), frameon=False, handlelength=1.4, handleheight=0.9,
               columnspacing=1.8)
    fig.supylabel("Pearson r with accuracy", fontsize=fs["label"])
    save_fig(fig, name)
    plt.show()
    return bars


plot_bars(["end"], "fig11_bars_end")
bars = plot_bars(["end", "mid"], "fig12_bars_end_mid")

In [ ]:
# Mean bar height over models x datasets, SEM over cells and over models
labels = {"baseline": "baseline", **PROBE_LABELS}
print(f"{'':<15} {'mean':>7} {'SEM(cells)':>11} {'SEM(models)':>12}")
for key, v in bars.items():
    v = np.array(v, dtype=float)
    print(f"{labels[key]:<15} {np.nanmean(v):>7.3f} {sem(v.ravel()):>11.3f} "
          f"{sem(np.nanmean(v, axis=1)):>12.3f}")

## Figs 13–16 — Scatter grids
Trained confidence vs accuracy and vs consistency, one panel per (model, dataset), first run of each configuration.

In [ ]:
SCATTER_RC = {"font.size": 7, "axes.titlesize": 8, "axes.labelsize": 9,
              "xtick.labelsize": 6, "ytick.labelsize": 6, "axes.linewidth": 0.5,
              "xtick.major.width": 0.5, "ytick.major.width": 0.5}
TARGETS = {"accuracy": acc, "consistency": cons}


def scatter_panel(ax, x, y, color):
    """Small density scatter + quadratic fit, r and R² inside the panel."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    ok = ~(np.isnan(x) | np.isnan(y))
    x, y = x[ok], y[ok]
    if x.size < 3:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes,
                color="0.6", fontsize=6)
        return
    density_scatter(ax, x, y, s=1.5, alpha=0.5)
    r2 = quadratic_fit(ax, x, y, color=color, lw=1.2)
    ax.text(0.03, 0.97, f"r={nan_pearsonr(x, y):.2f}\nR²={r2:.2f}", transform=ax.transAxes,
            ha="left", va="top", fontsize=5.5, linespacing=1.1, zorder=5,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75))


def scatter_grid(probe, target, name):
    """Rows = models, columns = datasets."""
    with plt.rc_context(SCATTER_RC):
        fig, axs = plt.subplots(len(MODELS), len(DATASETS), figsize=(7.9, 10.6),
                                sharex=True, sharey=True, layout="constrained")
        fig.get_layout_engine().set(w_pad=0.005, h_pad=0.005, wspace=0.01, hspace=0.01)
        for i, m in enumerate(MODELS):
            for j, d in enumerate(DATASETS):
                ax = axs[i, j]
                runs = trained[d][m][probe]
                if runs:
                    scatter_panel(ax, TARGETS[target][d][m], runs[0]["test"]["predictions"],
                                  COLORS[probe])
                else:
                    ax.text(0.5, 0.5, "no run", ha="center", va="center",
                            transform=ax.transAxes, color="0.6", fontsize=6)
                style_ax(ax, grid_axis="both")
                ax.tick_params(length=2, pad=1.5)
                ax.set_xticks([0, 0.5, 1.0])
                ax.set_yticks([0, 0.5, 1.0])
                ax.set_xlim(-0.03, 1.03)
                ax.set_ylim(-0.03, 1.03)
                if i == 0:
                    ax.set_title(SHORT_LABELS[d], pad=3, fontsize=8)
                if j == 0:
                    ax.set_ylabel(m, rotation=0, ha="right", va="center", fontsize=6, labelpad=5)
        fig.supxlabel(target, fontsize=10)
        fig.supylabel("trained confidence", fontsize=10)
        save_fig(fig, name)
    plt.show()


for num, (probe, target) in zip(range(13, 17), [("end", "accuracy"), ("end", "consistency"),
                                                 ("mid", "accuracy"), ("mid", "consistency")]):
    scatter_grid(probe, target, f"fig{num}_scatter_{probe}_{target}")